In [0]:
from pyspark.sql import functions as F

In [0]:
# --- Configuration ---
CATALOG = "iran_israel_capstone_project"
SCHEMA = "bronze"
LANDING_PATH = "abfss://capstonecontainer@iranisrael65.dfs.core.windows.net/landing_zone/fii_data/Merged_FII_Data.csv"

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

In [0]:
# --- 1. ADLS Landing Zone to Unity Catalog (Bronze) ---
print(f"Reading FII data from: {LANDING_PATH}")
df_fii_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(LANDING_PATH)

In [0]:
# Transformation & Mapping (DII Removed to match your previous logic)
df_fii_mapped = df_fii_raw.select(
    F.to_date(F.col("DATE")).alias("date"),
    F.col("`FII EQUITY Net Purchase / Sales`").cast("float").alias("fii_net_buy_sell_cr"),
    F.col("`FII EQUITY Gross Purchase`").cast("float").alias("fii_gross_buy_cr"),
    F.col("`FII EQUITY Gross Sales`").cast("float").alias("fii_gross_sell_cr")
)

In [0]:
# Add Audit Columns
df_fii_bronze = df_fii_mapped.withColumn("ingestion_timestamp", F.current_timestamp()) \
                             .withColumn("source_file", F.lit("Merged_FII_Data.csv"))

In [0]:
# Write to Unity Catalog
df_fii_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("fii_raw")
print("Successfully written to bronze.fii_raw")